# 09 · Seed aggregation, paired tests across seeds, and minimum detectable effects (no training)

Collects the per-seed result files written by notebooks 01, 02/03 and 04 and answers the review points on
seed count and statistical power:

* mean ± SD and a t-based 95 % interval per model, for single-pass and flip-TTA accuracy;
* **paired** comparison of SMGF-Net with every baseline / ablation arm across seeds (same seed = same split, same test set):
  mean difference, SD of the difference, paired t test, exact Wilcoxon signed-rank test (needs ≥ 5 seeds), seeds in which SMGF-Net is better;
* how many seeds had a significant exact McNemar test on the 995 test images;
* **minimum detectable differences**: across seeds (paired and unpaired) and on one fixed test set (McNemar approximation).

**Attach** the output folders of notebooks 01, 02 (and 03 if used) and 04 as Kaggle inputs; every file is looked up by name under
`/kaggle/input` and `/kaggle/working`; if several copies of a file are attached, the most complete one is used. No GPU is needed. Report only comparisons with at least 5 seeds (10 preferred).

In [ ]:
import glob, math, json
import numpy as np, pandas as pd
from pathlib import Path
from scipy import stats
OUT_DIR = Path("/kaggle/working"); OUT_DIR.mkdir(exist_ok=True, parents=True)

INPUT_ROOTS = ["/kaggle/input", str(OUT_DIR)]

def find_all(name):
    """The one best copy of a result file: names are matched ignoring spaces, and if several copies are attached
    (older partial runs) the one with the most rows / arrays is used, so partial files never mix in."""
    stem = name.replace(" ", "").lower(); hits = []
    for root in INPUT_ROOTS:
        hits += [p for p in Path(root).glob("**/*") if p.is_file() and p.name.replace(" ", "").lower() == stem]
    if not hits: return []
    size = (lambda p: len(pd.read_csv(p))) if name.endswith(".csv") else (lambda p: len(np.load(p, allow_pickle=True).files))
    best = max(hits, key=size); print(f"[input] {name}: {best}  ({size(best)} {'rows' if name.endswith('.csv') else 'arrays'})")
    return [best]

def read_all(name):
    frames = [pd.read_csv(p).assign(_source=str(p)) for p in find_all(name)]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

ms  = read_all("multiseed_results.csv")           # notebook 01: SMGF-Net, full model
bl  = read_all("baseline_results.csv")            # notebooks 02/03: SMGF-Net + baselines (columns model, seed)
ab  = read_all("plain_backbone_results.csv")      # notebook 04: ablation arms (columns arm, seed)
print({k: len(v) for k, v in {"multiseed (01)": ms, "baselines (02/03)": bl, "ablation arms (04)": ab}.items()})

## 1 · Per-model mean ± SD over seeds

In [ ]:
def t_interval(x, level=0.95):
    x = np.asarray(x, float); n = len(x)
    if n < 2: return (np.nan, np.nan)
    h = stats.t.ppf(0.5 + level / 2, n - 1) * x.std(ddof=1) / math.sqrt(n)
    return x.mean() - h, x.mean() + h

def summarise(df, key, cols):
    out = []
    for name, g in df.drop_duplicates([key, "seed"]).groupby(key):
        row = {key: name, "n_seeds": g.seed.nunique(), "seeds": sorted(g.seed.unique().tolist())}
        for c in cols:
            if c in g:
                lo, hi = t_interval(g[c]); row.update({f"{c}_mean": g[c].mean(), f"{c}_sd": g[c].std(ddof=1) if len(g) > 1 else np.nan,
                                                       f"{c}_lo": lo, f"{c}_hi": hi})
        out.append(row)
    return pd.DataFrame(out)

tables = {}
if len(bl):
    tables["baselines"] = summarise(bl, "model", ["acc_single", "acc_tta", "f1_macro_tta", "auc_tta", "ece_tta"])
if len(ab):
    tables["ablation"] = summarise(ab, "arm", ["acc_single", "acc_tta", "f1_macro_tta", "auc_tta", "ece_tta"])
if len(ms):
    m2 = ms.rename(columns={"accuracy_single": "acc_single", "accuracy_tta": "acc_tta"}).assign(model="SMGF-Net (notebook 01)")
    tables["notebook01"] = summarise(m2, "model", ["acc_single", "acc_tta", "accuracy_headline", "auc", "ece"])
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
for k, t in tables.items():
    t.to_csv(OUT_DIR / f"seed_summary_{k}.csv", index=False)
    print(f"\n[{k}]"); print(t.round(4).to_string(index=False))
short = [k for k, t in tables.items() if len(t) and t.n_seeds.min() < 5]
if short:
    print(f"\nWARNING: fewer than 5 seeds in: {short}. Do not claim an ordering for those models from this table.")

## 2 · Paired comparison across seeds (SMGF-Net vs each baseline / arm)
Pairing is by seed: the same seed gives the same split and the same test set. The test statistic uses the
per-seed accuracy difference, so seed-to-seed variation common to both models cancels.

In [ ]:
def paired_across_seeds(df, key, ref, col):
    wide = df.drop_duplicates([key, "seed"]).pivot(index="seed", columns=key, values=col)
    rows = []
    for other in [c for c in wide.columns if c != ref]:
        d = (wide[ref] - wide[other]).dropna()
        n = len(d)
        if n < 2: continue
        t_p = stats.ttest_1samp(d, 0.0).pvalue if n >= 3 else np.nan
        try: w_p = stats.wilcoxon(d).pvalue if n >= 5 and (d != 0).any() else np.nan
        except ValueError: w_p = np.nan
        lo, hi = t_interval(d)
        rows.append({"comparison": f"{ref} - {other}", "metric": col, "n_seeds": n, "mean_diff_pp": 100 * d.mean(),
                     "sd_diff_pp": 100 * d.std(ddof=1), "ci95_lo_pp": 100 * lo, "ci95_hi_pp": 100 * hi,
                     "paired_t_p": t_p, "wilcoxon_p": w_p, "seeds_ref_better": int((d > 0).sum()), "seeds_tied": int((d == 0).sum())})
    return pd.DataFrame(rows)

paired = []
if len(bl):
    for col in ("acc_single", "acc_tta"):
        paired.append(paired_across_seeds(bl, "model", "SMGF-Net + MS-CA-ECA", col))
if len(ab):
    full = "Full (SMGF-Net + MS-CA-ECA)"
    for col in ("acc_single", "acc_tta"):
        paired.append(paired_across_seeds(ab, "arm", full, col))
paired_df = pd.concat(paired, ignore_index=True) if paired else pd.DataFrame()
paired_df.to_csv(OUT_DIR / "paired_across_seeds.csv", index=False)
print(paired_df.round(4).to_string(index=False) if len(paired_df) else "no paired data found")
if len(paired_df):
    k = paired_df[["paired_t_p", "wilcoxon_p"]].notna().any(axis=1).sum()
    print(f"\n{len(paired_df)} comparisons were made; with a 0.05 level expect {0.05 * len(paired_df):.1f} to look 'significant' by chance. "
          "Apply a Holm correction before calling any single one significant (below).")
    ps = paired_df["paired_t_p"].fillna(1.0).to_numpy(); order = np.argsort(ps); holm = np.empty_like(ps)
    run = 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (len(ps) - rank) * ps[i])); holm[i] = run
    paired_df["paired_t_p_holm"] = holm; paired_df.to_csv(OUT_DIR / "paired_across_seeds.csv", index=False)
    print(paired_df[["comparison", "metric", "n_seeds", "mean_diff_pp", "paired_t_p", "paired_t_p_holm"]].round(4).to_string(index=False))

## 3 · McNemar on the fixed test set, per seed

In [ ]:
from scipy.stats import binomtest
rows = []
for name in ("baseline_predictions.npz", "plain_backbone_predictions.npz"):
    for p in find_all(name):
        z = np.load(p, allow_pickle=True); keys = [k for k in z.files if k.endswith("|y")]
        ref_name = "SMGF-Net + MS-CA-ECA" if name.startswith("baseline") else "Full (SMGF-Net + MS-CA-ECA)"
        for k in keys:
            arm, seed, _ = k.rsplit("|", 2)
            if arm == ref_name: continue
            rk = f"{ref_name}|{seed}|y"
            if rk not in z.files: continue
            y = z[rk]; ca = z[f"{ref_name}|{seed}|ptta"].argmax(1) == y; cb = z[f"{arm}|{seed}|ptta"].argmax(1) == y
            b, c = int((ca & ~cb).sum()), int((~ca & cb).sum())
            rows.append({"comparison": f"{ref_name} vs {arm}", "seed": int(seed), "ref_right_other_wrong": b, "ref_wrong_other_right": c,
                         "discordant_fraction": (b + c) / len(y), "mcnemar_exact_p": 1.0 if b + c == 0 else binomtest(min(b, c), b + c, 0.5).pvalue,
                         "n_test": len(y)})
mc = pd.DataFrame(rows)
if len(mc):
    mc.to_csv(OUT_DIR / "mcnemar_by_seed.csv", index=False)
    summ = mc.groupby("comparison").agg(n_seeds=("seed", "nunique"), seeds_p_below_0_05=("mcnemar_exact_p", lambda s: int((s < 0.05).sum())),
                                        mean_discordant=("discordant_fraction", "mean"), n_test=("n_test", "first")).reset_index()
    print(summ.round(4).to_string(index=False))
else:
    print("no prediction files found")

## 4 · Minimum detectable effects
(a) **Across seeds**: smallest true difference in mean accuracy that a paired t test over *n* seeds detects with 80 % power at α = 0.05,
    MDE = (t₍1−α/2, n−1₎ + t₍0.8, n−1₎)·s_d/√n, using the observed SD of the per-seed differences s_d.
(b) **Unpaired**, two groups of *k* seeds with the pooled single-model SD: MDE = (t₍1−α/2, 2k−2₎ + t₍0.8, 2k−2₎)·s·√(2/k).
(c) **One fixed test set** of N images: McNemar approximation MDE ≈ (z₁−α/2 + z₀.₈)·√(p_disc/N), p_disc = fraction of images on which the two models disagree.
These are planning numbers computed from the observed spread; they are not post-hoc power of an observed result.

In [ ]:
def mde_paired(sd_diff, n, alpha=0.05, power=0.8):
    return (stats.t.ppf(1 - alpha / 2, n - 1) + stats.t.ppf(power, n - 1)) * sd_diff / math.sqrt(n)
def mde_unpaired(sd, k, alpha=0.05, power=0.8):
    return (stats.t.ppf(1 - alpha / 2, 2 * k - 2) + stats.t.ppf(power, 2 * k - 2)) * sd * math.sqrt(2.0 / k)
def mde_mcnemar(p_disc, N, alpha=0.05, power=0.8):
    return (stats.norm.ppf(1 - alpha / 2) + stats.norm.ppf(power)) * math.sqrt(p_disc / N)

mde_rows = []
if len(paired_df):
    for _, r in paired_df[paired_df.metric == "acc_tta"].iterrows():
        sd = r.sd_diff_pp / 100
        for n in (3, 5, 10):
            mde_rows.append({"quantity": f"paired, {r.comparison}", "n_seeds": n, "mde_pp": 100 * mde_paired(sd, n)})
all_single = pd.concat([d for d in (bl.rename(columns={"model": "name"}), ab.rename(columns={"arm": "name"})) if len(d)], ignore_index=True) if (len(bl) or len(ab)) else pd.DataFrame()
if len(all_single):
    sd_pool = float(np.sqrt(all_single.groupby("name").acc_tta.var(ddof=1).dropna().mean()))
    for k in (3, 5, 10):
        mde_rows.append({"quantity": f"unpaired, pooled single-model SD {100 * sd_pool:.2f} pp", "n_seeds": k, "mde_pp": 100 * mde_unpaired(sd_pool, k)})
if len(mc):
    pd_ = float(mc.discordant_fraction.mean()); N = int(mc.n_test.iloc[0])
    mde_rows.append({"quantity": f"fixed test set, N = {N}, mean disagreement {100 * pd_:.1f} %", "n_seeds": np.nan, "mde_pp": 100 * mde_mcnemar(pd_, N)})
mde_df = pd.DataFrame(mde_rows); mde_df.to_csv(OUT_DIR / "minimum_detectable_effects.csv", index=False)
print(mde_df.round(3).to_string(index=False) if len(mde_df) else "nothing to compute yet")
print("\nState these in the paper next to every architecture / baseline claim: a difference smaller than the MDE for the number of seeds actually run is not a finding.")